# ---- smoke pass: every config, a few steps, small evaluations ----------------------------
SMOKE_EVAL = ("eval=[{name: test, manifest: manifests/clean_split/test.csv, limit: 64, "
              "subsets: {original: \"aug_type == 'original'\"}}, "
              "{name: plantdoc, manifest: manifests/plantdoc/test.csv, limit: 64, "
              "subsets: {exact: \"match == 'exact'\"}}]")
if SMOKE_TEST:
    smoke = WORK / "smoke"
    jobs = [(1, ["--config", c, "--set", *common, f"output_dir={smoke}", "seed=0",
                 "train.epochs=1", "train.steps_per_epoch=5", "train.validation_steps=2",
                 "train.save_model=false", SMOKE_EVAL]) for c in BATCHES[BATCH]]
    run_all(jobs, "smoke")
    ok = sorted(p.parent.parent.name for p in smoke.glob("*/seed0/metrics.json"))
    print("smoke OK:", ok)
    assert len(ok) == len(BATCHES[BATCH]), "smoke pass failed: see /kaggle/working/logs/smoke_gpu*.log"
    shutil.rmtree(smoke)

In [ ]:
# ---- settings -------------------------------------------------------------------------
REPO_URL = "https://github.com/MoazamAli45/plant-leaf-disease-detection.git"
BRANCH = "main"
BATCH = "main"          # "main" or "ablations"
SEEDS = [42, 43, 44]
SMOKE_TEST = True       # quick pass over every config before the real runs

BATCHES = {
    # config: relative cost (used to balance the two GPU queues)
    "main": {
        "configs/phase3/efficientnet_b0.yaml": 3,
        "configs/phase3/mobilenet_v1.yaml": 2,
        "configs/phase3/mobilenet_v2.yaml": 2,
        "configs/phase3/cnn_softmax.yaml": 2,
        "configs/phase3/mobilenet_v3_small.yaml": 1,
    },
    "ablations": {
        "configs/phase3/abl_mnv1_noaug.yaml": 2,
        "configs/phase3/abl_mnv1_frozen.yaml": 1,
        "configs/phase3/abl_mnv1_alpha075.yaml": 2,
        "configs/phase3/abl_mnv1_alpha050.yaml": 1,
    },
}
# weights kept for Grad-CAM / efficiency (seed 42 only)
KEEP_MODELS = ["p3_cnn_softmax", "p3_mobilenet_v1", "p3_mobilenet_v2", "p3_mobilenet_v3_small",
               "p3_efficientnet_b0", "p3_abl_mnv1_alpha050", "p3_abl_mnv1_alpha075",
               "p3_abl_mnv1_frozen"]

In [ ]:
import glob, json, os, shutil, subprocess, sys, threading, time
from pathlib import Path

WORK = Path("/kaggle/working")
repo = WORK / "repo"
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
ML = repo / "ml"
os.chdir(ML)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print("code at", ML, "commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
      capture_output=True, text=True).stdout.strip())

# NPD (attached Kaggle dataset)
cands = [Path(p).parent for p in glob.glob("/kaggle/input/**/train", recursive=True)
         if (Path(p).parent / "valid").is_dir() and len(list(Path(p).iterdir())) == 38]
assert cands, "Attach the 'New Plant Diseases Dataset' (vipoooool) as an input."
NPD_ROOT = sorted(cands, key=str)[0]
print("NPD root:", NPD_ROOT)

# PlantDoc (public GitHub; same pinned commit and file layout as the manifests)
subprocess.run([sys.executable, "-m", "leafdx.data.download", "--data-dir", "data",
                "--only", "plantdoc"], check=True)
print("PlantDoc files:", sum(1 for _ in Path("data/raw/plantdoc").rglob("*.*")))

In [ ]:
import tensorflow as tf, keras
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPUs:", len(gpus))
assert gpus, "No GPU: enable one under Settings -> Accelerator."
N_GPU = len(gpus)

def queues(jobs):
    """Split (cost, args) jobs over the GPUs, longest first."""
    qs, load = [[] for _ in range(N_GPU)], [0] * N_GPU
    for cost, args in sorted(jobs, key=lambda j: -j[0]):
        k = load.index(min(load)); qs[k].append(args); load[k] += cost
    return qs

def run_queue(gpu, jobs, log_path):
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "TF_CPP_MIN_LOG_LEVEL": "2"}
    with open(log_path, "a") as log:
        for args in jobs:
            t = time.time()
            log.write(f"\n>>> {' '.join(args)}\n"); log.flush()
            r = subprocess.run([sys.executable, "-m", "leafdx.train.run", *args],
                               env=env, stdout=log, stderr=subprocess.STDOUT)
            msg = f"[gpu{gpu}] {'OK ' if r.returncode == 0 else 'FAILED'} {args[1]} {args[-1]} " \
                  f"in {(time.time() - t) / 60:.1f} min"
            log.write(msg + "\n"); log.flush(); print(msg, flush=True)

def run_all(jobs, tag):
    logs = WORK / "logs"; logs.mkdir(exist_ok=True)
    threads = [threading.Thread(target=run_queue, args=(g, q, logs / f"{tag}_gpu{g}.log"))
               for g, q in enumerate(queues(jobs))]
    for th in threads: th.start()
    for th in threads: th.join()

common = [f"data.npd_root={NPD_ROOT}"]

In [ ]:
# ---- smoke pass: every config, a few steps, small evaluations ----------------------------
if SMOKE_TEST:
    smoke = WORK / "smoke"
    jobs = [(1, ["--config", c, "--set", *common, f"output_dir={smoke}", "train.epochs=1",
                 "train.steps_per_epoch=5", "train.validation_steps=2", "train.save_model=false",
                 "eval[0].limit=64", "seed=0"]) for c in BATCHES[BATCH]]
    # eval[*].limit is not a valid override path; patch limits via a tiny YAML instead
    for _, a in jobs:
        a.remove("eval[0].limit=64")
        a.append("eval=[{name: test, manifest: manifests/clean_split/test.csv, limit: 64, "
                 "subsets: {original: \"aug_type == 'original'\"}}, {name: plantdoc, "
                 "manifest: manifests/plantdoc/test.csv, limit: 64, subsets: {exact: \"match == 'exact'\"}}]")
    run_all(jobs, "smoke")
    ok = sorted(p.parent.parent.name for p in smoke.glob("*/seed0/metrics.json"))
    print("smoke OK:", ok)
    assert len(ok) == len(BATCHES[BATCH]), "smoke pass failed: see /kaggle/working/logs/smoke_gpu*.log"
    shutil.rmtree(smoke)

In [ ]:
# ---- the real runs ----------------------------------------------------------------------
t0 = time.time()
jobs = [(cost, ["--config", c, "--set", *common, f"output_dir={WORK / 'outputs'}", f"seed={s}"])
        for c, cost in BATCHES[BATCH].items() for s in SEEDS]
run_all(jobs, BATCH)
print(f"all runs finished in {(time.time() - t0) / 3600:.2f} h")

In [ ]:
# ---- pack results (all runs) and seed-42 weights ---------------------------------------
subprocess.run([sys.executable, "-m", "leafdx.train.report", "collect", str(WORK / "outputs"),
                "--results", str(WORK / "results")], check=True)
shutil.copytree(WORK / "logs", WORK / "results" / f"_logs_{BATCH}", dirs_exist_ok=True)
subprocess.run(["zip", "-qr", str(WORK / f"results_{BATCH}.zip"), "results"], cwd=WORK, check=True)

models = WORK / "models"
for name in KEEP_MODELS:
    src = WORK / "outputs" / name / "seed42" / "model.keras"
    if src.exists():
        (models / name).mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, models / name / "model.keras")
        shutil.copy2(src.parent / "config.yaml", models / name / "config.yaml")
if models.exists():
    subprocess.run(["zip", "-qr", str(WORK / f"models_{BATCH}.zip"), "models"], cwd=WORK, check=True)
shutil.rmtree(WORK / "outputs")  # keep the output small: weights are in models_*.zip
for m in sorted((WORK / "results").glob("p3_*/seed*/metrics.json")):
    d = json.loads(m.read_text()); e = d["eval"]
    print(f"{d['name']:24s} s{d['seed']}  test={e['test']['accuracy']:.4f}  "
          f"plantdoc={e['plantdoc']['accuracy']:.4f}  best_epoch={d['best_epoch']}")
print("download:", sorted(p.name for p in WORK.glob("*.zip")))